# Single vs multi-agent matching on rjdfit (Colab)

Runs the evaluation described in the README with a local model through Ollama (set `MODEL` and `CONFIG` in the second code cell; this version uses Qwen3 14B, 4-bit) on a Colab GPU:

1. **Data:** downloads the public rjdfit dataset and builds 30 resume pools (2 dev, 28 eval).
2. **Dev check:** both pipelines on the 2 dev resumes. Every run must succeed before the eval starts.
3. **Eval:** both pipelines on the 28 eval resumes.
4. **Score:** nDCG@5, precision@5, top-1 hit rate, latency and tokens, plus a paired comparison per resume.

**Before you start:** push your latest code to GitHub (the notebook clones `origin/main`), pick **Runtime > Change runtime type > GPU**
(L4 or A100 is fastest), then **Runtime > Run all**. Results are saved to Google Drive after every run. If Colab disconnects,
reconnect and **Run all** again: the runner skips runs that are already recorded.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: choose Runtime > Change runtime type > GPU"

In [ ]:
# Keep results on Google Drive so a disconnect does not lose completed runs.
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/agentic_resume_matcher_rjdfit_14b'   # new folder so the earlier 4B results stay untouched
MODEL = 'qwen3:14b-q4_K_M'
CONFIG = 'shared/local_14b_experiment.json'
CONTEXT = 40960   # Qwen3 14B's native window
DEV, EVAL = f'{RESULTS}/dev', f'{RESULTS}/eval'
REPETITIONS = 1   # temperature is 0 with a fixed seed, so extra repetitions mostly repeat the same output
!mkdir -p "{RESULTS}"

In [ ]:
%cd /content
!test -d agentic_resume_matcher || git clone https://github.com/RitvikNeerattil/agentic_resume_matcher.git
%cd /content/agentic_resume_matcher
# Pin the code version on first start so resumed sessions run identical code.
import os
pin = f'{RESULTS}/code_commit.txt'
!git fetch -q origin
if os.path.exists(pin):
    commit = open(pin).read().strip()
else:
    commit = !git rev-parse origin/main
    commit = commit[0]
    open(pin, 'w').write(commit)
!git checkout -q {commit} && git log -1 --format="Code version: %h %s"

In [ ]:
# Install and start Ollama (flash attention + 8-bit context cache to fit 131k tokens on the GPU).
import os, subprocess, time, urllib.request
!command -v ollama >/dev/null || (apt-get -qq install -y zstd >/dev/null && curl -fsSL https://ollama.com/install.sh | sh)
env = {**os.environ, 'OLLAMA_FLASH_ATTENTION': '1', 'OLLAMA_KV_CACHE_TYPE': 'q8_0', 'OLLAMA_NUM_PARALLEL': '1'}
subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/version'); print('Ollama is running'); break
    except Exception:
        time.sleep(1)
!ollama pull {MODEL}

In [ ]:
# Offline tests, then a one-token request at the full context size to confirm it fits on the GPU.
!pip install -q datasets
!python -m unittest shared.test_workflow shared.test_rjdfit
import json, urllib.request
body = {'model': MODEL, 'prompt': 'hi', 'stream': False,
        'options': {'num_ctx': CONTEXT, 'num_predict': 1}}
urllib.request.urlopen(urllib.request.Request('http://127.0.0.1:11434/api/generate', json.dumps(body).encode(),
                                              {'Content-Type': 'application/json'}), timeout=600).read()
!ollama ps
print('PROCESSOR should read "100% GPU". A CPU/GPU split still works, but more slowly.')

## 1. Data: download rjdfit and build the resume pools

In [ ]:
!python -m shared.rjdfit_eval download
!python -m shared.rjdfit_eval build

## 2. Dev check (2 resumes)

In [ ]:
!python -m shared.rjdfit_eval run --config {CONFIG} --split dev --repetitions {REPETITIONS} --output "{DEV}" --execute --resume
runs = [json.loads(l) for l in open(f'{DEV}/runs.jsonl')]
for r in runs:
    print(r['resume_id'], r['size'], f"{r['architecture']:12}", r['status'], f"{r['seconds']:.0f}s")
failed = [r for r in runs if r['status'] != 'ok']
assert runs and not failed, 'Dev runs must all succeed before the eval. Fix the failure, then rename or delete the dev folder in Drive and rerun this cell.'
print('Dev check passed')

## 3. Eval (28 resumes)

In [ ]:
!python -m shared.rjdfit_eval run --config {CONFIG} --split eval --repetitions {REPETITIONS} --output "{EVAL}" --execute --resume

## 4. Results

In [ ]:
!python -m shared.rjdfit_eval summarize "{EVAL}" > /dev/null
import pandas as pd
display(pd.read_csv(f'{EVAL}/comparison.csv').round(3))
paired = pd.read_csv(f'{EVAL}/paired_resumes.csv')
display(paired.round(3))
d = paired['difference']
print(f'Orchestrated minus single, nDCG@5: mean {d.mean():+.3f}; better on {(d>0).sum()}, worse on {(d<0).sum()}, tied on {(d==0).sum()} of {len(d)} resumes')

In [ ]:
# Download the summaries (no raw model responses, which contain dataset text).
import shutil, os
out = '/content/rjdfit_summary'
os.makedirs(out, exist_ok=True)
for name in ('comparison.csv', 'paired_resumes.csv', 'manifest.json'):
    if os.path.exists(f'{EVAL}/{name}'):
        shutil.copy(f'{EVAL}/{name}', out)
shutil.make_archive('/content/rjdfit_summary', 'zip', out)
from google.colab import files
files.download('/content/rjdfit_summary.zip')
print('The full raw runs stay in Google Drive:', RESULTS)